# TradeSense · Component 03 · notebook 47: the "UNSEEN PATTERN" signal of the Autoencoder, Part A (CPU, about 3 to 8 minutes)

**Question.** XGBoost and Transformer recognise known abnormal patterns; the Autoencoder is label-free. When the AE flags a minute while XGBoost and Transformer do not, does that signal catch different events (including families that are hard for the trained models), at a small genuine alarm cost?

**What this notebook does.** It does NOT retrain or re-score anything. It reads the TEST-B / TEST-3 scores that notebook 32 saved (`scores/test_b_*.npz`), checks that they are the same files notebook 32 recorded (sha), reproduces notebook 32's own headline numbers (so the definitions are identical), and then evaluates protocol U1 v2 (Part A) with hypotheses H-U1..H-U3 that were written before any number existed. TEST-3 was opened once in T0, so this is reported as a **new pre-registered analysis on saved scores**; nothing is retuned and nothing frozen changes.

Part B (the leave-one-family-out test that gives the truly unseen evidence) is notebook 48.

**The protocol text hashed by this notebook (frozen when this cell runs):**


```
# Protocol U1 v2 (approved by the user 2026-10-16; frozen when notebook 47 hashes this exact text) - Known + previously unseen abnormality: the "UNSEEN PATTERN" signal of the Autoencoder
Written BEFORE any result of this study exists. Nothing frozen (models, 6 features, normalisation, fusion stacker, thresholds q) is changed. XGBoost, Transformer, AE, fusion, Integrity Score and states stay exactly as they are. The signal below is an ADDITIONAL output.

## Idea (the user's novelty 2)
XGBoost and Transformer recognise KNOWN abnormal patterns. The Autoencoder, trained on normal behaviour only (label-free), flags behaviour it has not seen. When the AE flags a minute while XGBoost and Transformer do not, the dashboard shows a badge "UNSEEN PATTERN" next to the normal state.

## The signal (rule declared now)
UNSEEN PATTERN at bar t = the AE (A6, mean10, N1) is above its own T1 rolling alert line at budget 2/day AND XGBoost and Transformer are both below their own T1 lines (budget 2/day each). Source: the `agree` bit mask of the engine (AE only = 1). No new threshold, no new fitting. Wording on the screen: "Only the label-free Autoencoder flagged this; the trained models did not recognise it. It is a statistical signal, not proof of a new manipulation type."

## Part A - signal on saved TEST-3 scores (no training)
New pre-registered analysis on saved TEST-3 scores (TEST-3 was opened once in T0; nothing is retuned; library sha must be 3b5d9df23521adf9). Per family: three-way split of detected events (AE only / supervised only / both / neither), genuine TEST-B episodes/day of the signal, Sep vs Oct, per symbol.
- H-U1: genuine TEST-B burden of the UNSEEN PATTERN signal (episodes/day) is <= 1.0.
- H-U2 (supervised dominates on known families): on the six known families, recall(XGBoost or Transformer at 2/day) - recall(AE at 2/day) >= 0.20.
- H-U3 (genuinely different catches): macro over all 8 families, at least 15% of the events caught by the AE are missed by fusion.

## Part B - the real "unseen" test: leave-one-family-out (LOFO) system study (VAL-3 for design, CAL-3 reported once)
For each of the six known families f: train a STUDY copy of XGBoost (same config as the frozen v2: A | pooled | depth 3, seeds 11/22/33, N1) on TRAIN-3 WITHOUT family f; the frozen models are untouched and not replaced. Evaluate on family f (T1c, 2/day, mean BTC/ETH, same windows and episodes as notebook 29b): S_wo = XGBoost without f; N = AE (A6); "S_wo or N" = event detected if either alarms; genuine burden of the OR reported.
- H-U4 (the AE adds on a truly unseen family): macro over the six families, recall(S_wo or N) - recall(S_wo) >= 0.05 with paired event-bootstrap 95% interval above 0, on CAL-3.
- H-U5 (burden): genuine episodes/day of "S_wo or N" <= S_wo's + 2.1 (the AE's own budget plus its observed rate).
- Descriptive per family. Transformer LOFO (GPU) is a later extension, not part of this protocol version.

## Part C (conditional, only if H-U4 fails) - AE v3 as a NEW version
New folder, A6 AE kept; selection by family-level cross-validation on VAL-3; test on a freshly injected set with new seeds on the TEST-B genuine bars ("TEST-3b"), declared before it is built. Separate protocol.

## Part D (separate, after the data format is checked): real external evidence
Layer 2 (Sapienza pump-and-dump, Mt.Gox) with the domain shift stated; claimed only if the frozen features can be built from that data.

## Known weaknesses, stated in advance
AE alone is weak (38.2% macro-8 at 2/day in TEST-3), so H-U3 or H-U4 may fail and will be reported as failed. The two "new" families (momentum_ignition, slow_accumulation) share mechanics with trained families, so they are not a clean unseen test, which is why Part B exists. Labels are synthetic mechanisms. LOFO with XGBoost only first.

## Operational readings written before running notebook 47 (Part A)
- Signal bar = AE alarm bar AND NOT XGBoost alarm bar AND NOT Transformer alarm bar, each by the T1c rule (rolling genuine-series threshold, q frozen per budget 2 and symbol). Episodes merge bars within 5 minutes.
- "XGBoost or Transformer" = an event is detected if XGBoost (own 2/day rule) or Transformer (own 2/day rule) alarms inside the event window. "Caught by the AE" = the AE (2/day rule) alarms inside the event window; "missed by fusion" = fusion (2/day) does not.
- H-U3: for each family, share_f = (events caught by the AE and missed by fusion) / (events caught by the AE), pooled over both symbols; macro = mean over the families with at least 5 events caught by the AE (the others are listed, not averaged).
- Descriptive only: budgets 1 and 4 (the AE budget 0.5 does not exist in declared_ae_v2.json and is dropped), per symbol, Sep versus Oct, chance level of the signal (alarms rolled in time).

```

In [ ]:
import os, sys, json, time, hashlib, warnings, datetime, glob
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
L3 = BASE/'layer3_v3'; OUT = BASE/'models_b2_v1'
SYMBOLS = ['BTCUSDT', 'ETHUSDT']
MOCK = os.environ.get('TS_MOCK') == '1'; NBOOT = 200 if MOCK else 2000
LINES = []
def say(s=''): print(s); LINES.append(str(s))
def sha_file(p, n=16): return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:n]
PROTO_U1 = r"""# Protocol U1 v2 (approved by the user 2026-10-16; frozen when notebook 47 hashes this exact text) - Known + previously unseen abnormality: the "UNSEEN PATTERN" signal of the Autoencoder
Written BEFORE any result of this study exists. Nothing frozen (models, 6 features, normalisation, fusion stacker, thresholds q) is changed. XGBoost, Transformer, AE, fusion, Integrity Score and states stay exactly as they are. The signal below is an ADDITIONAL output.

## Idea (the user's novelty 2)
XGBoost and Transformer recognise KNOWN abnormal patterns. The Autoencoder, trained on normal behaviour only (label-free), flags behaviour it has not seen. When the AE flags a minute while XGBoost and Transformer do not, the dashboard shows a badge "UNSEEN PATTERN" next to the normal state.

## The signal (rule declared now)
UNSEEN PATTERN at bar t = the AE (A6, mean10, N1) is above its own T1 rolling alert line at budget 2/day AND XGBoost and Transformer are both below their own T1 lines (budget 2/day each). Source: the `agree` bit mask of the engine (AE only = 1). No new threshold, no new fitting. Wording on the screen: "Only the label-free Autoencoder flagged this; the trained models did not recognise it. It is a statistical signal, not proof of a new manipulation type."

## Part A - signal on saved TEST-3 scores (no training)
New pre-registered analysis on saved TEST-3 scores (TEST-3 was opened once in T0; nothing is retuned; library sha must be 3b5d9df23521adf9). Per family: three-way split of detected events (AE only / supervised only / both / neither), genuine TEST-B episodes/day of the signal, Sep vs Oct, per symbol.
- H-U1: genuine TEST-B burden of the UNSEEN PATTERN signal (episodes/day) is <= 1.0.
- H-U2 (supervised dominates on known families): on the six known families, recall(XGBoost or Transformer at 2/day) - recall(AE at 2/day) >= 0.20.
- H-U3 (genuinely different catches): macro over all 8 families, at least 15% of the events caught by the AE are missed by fusion.

## Part B - the real "unseen" test: leave-one-family-out (LOFO) system study (VAL-3 for design, CAL-3 reported once)
For each of the six known families f: train a STUDY copy of XGBoost (same config as the frozen v2: A | pooled | depth 3, seeds 11/22/33, N1) on TRAIN-3 WITHOUT family f; the frozen models are untouched and not replaced. Evaluate on family f (T1c, 2/day, mean BTC/ETH, same windows and episodes as notebook 29b): S_wo = XGBoost without f; N = AE (A6); "S_wo or N" = event detected if either alarms; genuine burden of the OR reported.
- H-U4 (the AE adds on a truly unseen family): macro over the six families, recall(S_wo or N) - recall(S_wo) >= 0.05 with paired event-bootstrap 95% interval above 0, on CAL-3.
- H-U5 (burden): genuine episodes/day of "S_wo or N" <= S_wo's + 2.1 (the AE's own budget plus its observed rate).
- Descriptive per family. Transformer LOFO (GPU) is a later extension, not part of this protocol version.

## Part C (conditional, only if H-U4 fails) - AE v3 as a NEW version
New folder, A6 AE kept; selection by family-level cross-validation on VAL-3; test on a freshly injected set with new seeds on the TEST-B genuine bars ("TEST-3b"), declared before it is built. Separate protocol.

## Part D (separate, after the data format is checked): real external evidence
Layer 2 (Sapienza pump-and-dump, Mt.Gox) with the domain shift stated; claimed only if the frozen features can be built from that data.

## Known weaknesses, stated in advance
AE alone is weak (38.2% macro-8 at 2/day in TEST-3), so H-U3 or H-U4 may fail and will be reported as failed. The two "new" families (momentum_ignition, slow_accumulation) share mechanics with trained families, so they are not a clean unseen test, which is why Part B exists. Labels are synthetic mechanisms. LOFO with XGBoost only first.

## Operational readings written before running notebook 47 (Part A)
- Signal bar = AE alarm bar AND NOT XGBoost alarm bar AND NOT Transformer alarm bar, each by the T1c rule (rolling genuine-series threshold, q frozen per budget 2 and symbol). Episodes merge bars within 5 minutes.
- "XGBoost or Transformer" = an event is detected if XGBoost (own 2/day rule) or Transformer (own 2/day rule) alarms inside the event window. "Caught by the AE" = the AE (2/day rule) alarms inside the event window; "missed by fusion" = fusion (2/day) does not.
- H-U3: for each family, share_f = (events caught by the AE and missed by fusion) / (events caught by the AE), pooled over both symbols; macro = mean over the families with at least 5 events caught by the AE (the others are listed, not averaged).
- Descriptive only: budgets 1 and 4 (the AE budget 0.5 does not exist in declared_ae_v2.json and is dropped), per symbol, Sep versus Oct, chance level of the signal (alarms rolled in time).
"""
PHU = hashlib.sha256(PROTO_U1.encode()).hexdigest(); (OUT/'protocol_u1_v2.md').write_text(PROTO_U1); (OUT/'protocol_u1_v2.sha256').write_text(PHU + '  protocol_u1_v2.md\n')
for f in ['test_opened_v1.json', 'test_b_results_v1.json', 'declared_final_set_v1.json', 'declared_ae_v2.json', 'declared_fusion_v0.json', 'declared_watch_v1.json', 'b2_lib.py', 'c03_infer.py']:
    assert (OUT/f).exists(), f'{f} missing: notebook 32 must have been run to the end first'
for s in SYMBOLS: assert (OUT/'scores'/f'test_b_{s}.npz').exists(), 'saved TEST scores missing'
T32 = json.loads((OUT/'test_b_results_v1.json').read_text())
for s in SYMBOLS: assert sha_file(OUT/'scores'/f'test_b_{s}.npz') == T32['test_score_files_sha16'][s], f'saved TEST scores of {s} differ from what notebook 32 recorded'
assert sha_file(OUT/'c03_infer.py') == T32['library_sha16'] and sha_file(OUT/'b2_lib.py') == T32['b2_lib_sha16'], 'library files differ from the ones that opened TEST'
sys.path.insert(0, str(OUT)); import importlib; import b2_lib as B; import c03_infer as c
say('protocol U1 v2 sha256 ' + PHU[:16] + '; notebook-32 score files and library match what was recorded at the TEST opening (library ' + T32['library_sha16'] + (')  [MOCK RUN]' if MOCK else ')'))

Mounted at /content/drive
protocol U1 v2 sha256 c448d8d48815aab0; notebook-32 score files and library match what was recorded at the TEST opening (library 3b5d9df23521adf9)


## 1 · Saved TEST scores and events

In [ ]:
AOLEN = 31 * 1440; AOFULL = 132480; TEST_LO = AOLEN; SEP_HI = TEST_LO + 30 * 1440
TV = np.zeros(AOFULL, bool); TV[TEST_LO:] = True; TV[:B.WARM] = False
TVS = TV.copy(); TVS[SEP_HI:] = False; TVO = TV.copy(); TVO[:SEP_HI] = False
DF = json.load(open(OUT/'declared_final_set_v1.json')); AE2 = json.load(open(OUT/'declared_ae_v2.json')); FUSD = json.load(open(OUT/'declared_fusion_v0.json'))
MODELS = c.MODEL_NAMES; TAG = dict(zip(MODELS, ['AE', 'XGB', 'TR']))
Q = {MODELS[0]: AE2['q_T1_by_budget'], MODELS[1]: DF['models'][MODELS[1]]['q_T1_by_budget'], MODELS[2]: DF['models'][MODELS[2]]['q_T1_by_budget'], 'Fusion': FUSD['q_T1_by_budget']}
SC = {}; EVT = {}; REPS = {}
for s in SYMBOLS:
    z = np.load(OUT/'scores'/f'test_b_{s}.npz'); keys = set(z.files); REPS[s] = sorted(int(k.split('__r')[1]) for k in keys if k.startswith('AE__r'))
    for m in MODELS:
        SC[(m, s, 'GEN')] = z[TAG[m] + '__GEN'].astype('float64')
        for r in REPS[s]: SC[(m, s, f'r{r}')] = z[f'{TAG[m]}__r{r}'].astype('float64')
    SC[('Fusion', s, 'GEN')] = z['FUS__GEN'].astype('float64')
    for r in REPS[s]: SC[('Fusion', s, f'r{r}')] = z[f'FUS__r{r}'].astype('float64')
    EVT[s] = pd.read_csv(L3/f'l3v3_{s}_TEST_events.csv'); assert EVT[s].t0.min() >= TEST_LO and (EVT[s].t0 + EVT[s].dur).max() <= AOFULL
say('saved TEST scores loaded: replicas ' + str(REPS) + '; TEST-3 events per symbol ' + str({s: len(EVT[s]) for s in SYMBOLS}) + '; genuine TEST-B days ' + str(int(TV.sum() // 1440)))

saved TEST scores loaded: replicas {'BTCUSDT': [0], 'ETHUSDT': [0]}; TEST-3 events per symbol {'BTCUSDT': 300, 'ETHUSDT': 300}; genuine TEST-B days 61


## 2 · Evaluation helpers (same definitions as notebook 32)

In [ ]:
NAMES = MODELS + ['Fusion']; OLD, NEW = [1, 2, 3, 4, 5, 6], [7, 8]; ALLF = OLD + NEW; PRIM = B.PRIMARY
THR = {}
def qof(name, b, s): return Q[name][str(b)][s]
def thr(name, s, q):
    k = (name, s, q)
    if k not in THR: THR[k] = c.rolling_threshold(SC[(name, s, 'GEN')], q)
    return THR[k]
def al_gen(name, s, b): return SC[(name, s, 'GEN')] > thr(name, s, qof(name, b, s))
def al_inj(name, s, key, b): return SC[(name, s, key)] > thr(name, s, qof(name, b, s))
SETS = [(s, r, EVT[s][EVT[s].replica == r].reset_index(drop=True)) for s in SYMBOLS for r in REPS[s]]
META = pd.concat([pd.DataFrame({'sym': s, 'rep': r, 'fam': ev.family.to_numpy(), 't0': ev.t0.to_numpy()}) for s, r, ev in SETS], ignore_index=True)
def event_hits(a, ev):
    cs = np.concatenate([[0], np.cumsum(a.astype(np.int64))]); t0 = ev.t0.to_numpy(); du = ev.dur.to_numpy(); return (cs[t0 + du] - cs[t0]) > 0
def alarm_gen(who, s, b):
    if who == 'signal': return al_gen(MODELS[0], s, b) & ~al_gen(MODELS[1], s, b) & ~al_gen(MODELS[2], s, b)
    if who == 'sup': return al_gen(MODELS[1], s, b) | al_gen(MODELS[2], s, b)
    return al_gen(who, s, b)
def alarm_inj(who, s, key, b):
    if who == 'signal': return al_inj(MODELS[0], s, key, b) & ~al_inj(MODELS[1], s, key, b) & ~al_inj(MODELS[2], s, key, b)
    if who == 'sup': return al_inj(MODELS[1], s, key, b) | al_inj(MODELS[2], s, key, b)
    return al_inj(who, s, key, b)
HC = {}
def hits(who, b):
    k = (who, b)
    if k not in HC: HC[k] = np.concatenate([event_hits(alarm_inj(who, s, f'r{r}', b), ev) for s, r, ev in SETS])
    return HC[k]
def gen_eps(who, b, valid=None):
    v = TV if valid is None else valid; return float(np.mean([B.eps_per_day(alarm_gen(who, s, b), v) for s in SYMBOLS]))
def macro_of(h, fams, sel=None, syms=None):
    out = []
    for s in (syms or SYMBOLS):
        k = ((META.sym == s) & META.fam.isin(fams)).to_numpy().copy()
        if sel is not None: k &= sel
        out.append(pd.DataFrame({'fam': META.fam[k].to_numpy(), 'h': h[k]}).groupby('fam').h.mean().mean())
    return float(np.mean(out))
GROUPS = [[np.flatnonzero(((META.sym == s) & (META.fam == f)).to_numpy()) for f in ALLF] for s in SYMBOLS]
def boot(h_list, fams, n=None, seed=0):
    n = n or NBOOT; rg = np.random.default_rng(seed); out = np.zeros((n, len(h_list))); fi = [ALLF.index(f) for f in fams]
    for i in range(n):
        acc = []
        for gs in GROUPS:
            fm = []
            for j in fi:
                idx = gs[j]; pick = idx[rg.integers(0, len(idx), len(idx))]; fm.append([h[pick].mean() for h in h_list])
            acc.append(np.mean(fm, axis=0))
        out[i] = np.mean(acc, axis=0)
    return out
def chance(who, b, fams, nsh=40, seed=0):
    rg = np.random.default_rng(seed); out = []
    for _ in range(nsh):
        hs = []
        for s, r, ev in SETS:
            seg = alarm_gen(who, s, b)[TEST_LO:]; full = np.zeros(AOFULL, bool); full[TEST_LO:] = np.roll(seg, int(rg.integers(0, len(seg)))); hs.append(event_hits(full, ev))
        out.append(macro_of(np.concatenate(hs), fams))
    return float(np.mean(out))
def pct(x): return f'{100 * x:.1f}%'
say('evaluation helpers ready (same definitions as notebook 32: T1c rolling genuine-series threshold on the injected series, q frozen per symbol and budget, episodes merge bars within 5 minutes)')

evaluation helpers ready (same definitions as notebook 32: T1c rolling genuine-series threshold on the injected series, q frozen per symbol and budget, episodes merge bars within 5 minutes)


## 3 · Unit tests and parity with notebook 32

In [ ]:
# T01 parity with notebook 32: the same definitions on the same saved scores must reproduce notebook 32's recorded headline numbers
WHO32 = {MODELS[0]: MODELS[0], MODELS[1]: MODELS[1], MODELS[2]: MODELS[2], 'FUSION (Integrity Score)': 'Fusion'}
for lab, who in WHO32.items():
    got = round(macro_of(hits(who, PRIM), ALLF), 4); want = T32['macro8_budget2'][lab]; assert abs(got - want) < 1.5e-4, (lab, got, want)
    ge = round(gen_eps(who, PRIM), 3); wg = T32['genuine_eps_day_budget2'][lab]; assert abs(ge - wg) < 2e-3, (lab, ge, wg)
say('PASS T01 notebook 32 headline numbers reproduced from the saved scores (macro-8 recall and genuine episodes/day at budget 2 for the three models and fusion)')
# T02 event hits equal a plain loop
s0, r0, ev0 = SETS[0]; a0 = alarm_inj('signal', s0, f'r{r0}', PRIM); h0 = event_hits(a0, ev0)
plain = np.array([bool(a0[t:t + d].any()) for t, d in zip(ev0.t0.to_numpy(), ev0.dur.to_numpy())]); assert (h0 == plain).all()
say('PASS T02 event hits of the signal equal a plain loop (first TEST-3 set)')
# T03 definition of the signal: bar-level truth table on a tiny synthetic case
ae_ = np.array([1, 1, 0, 1, 0], bool); x_ = np.array([0, 1, 0, 0, 0], bool); t_ = np.array([0, 0, 0, 1, 1], bool); sig_ = ae_ & ~x_ & ~t_; assert sig_.tolist() == [True, False, False, False, False]
say('PASS T03 signal = AE alarm and not XGBoost alarm and not Transformer alarm (truth table)')
# T04 the signal is a subset of the AE alarms and cannot overlap the supervised alarms
for s in SYMBOLS:
    sg = alarm_gen('signal', s, PRIM); assert (sg <= al_gen(MODELS[0], s, PRIM)).all() and not (sg & alarm_gen('sup', s, PRIM)).any()
say('PASS T04 the signal is a subset of the AE alarms and never overlaps an XGBoost or Transformer alarm bar')
# T05 no alarms before 1,440 bars of history; thresholds use genuine series only
for s in SYMBOLS: assert not alarm_gen('signal', s, PRIM)[:1440].any()
say('PASS T05 no alarm before 1,440 bars of history')
fam_ct = META.groupby(['sym', 'fam']).size().unstack(); assert fam_ct.shape[1] == 8 and fam_ct.min().min() >= 5; say('PASS T06 events of all 8 families in both symbols (min per symbol and family ' + str(int(fam_ct.min().min())) + ')')

PASS T01 notebook 32 headline numbers reproduced from the saved scores (macro-8 recall and genuine episodes/day at budget 2 for the three models and fusion)
PASS T02 event hits of the signal equal a plain loop (first TEST-3 set)
PASS T03 signal = AE alarm and not XGBoost alarm and not Transformer alarm (truth table)
PASS T04 the signal is a subset of the AE alarms and never overlaps an XGBoost or Transformer alarm bar
PASS T05 no alarm before 1,440 bars of history
PASS T06 events of all 8 families in both symbols (min per symbol and family 37)


## 4 · Results (protocol U1 v2, Part A)

In [ ]:
_nm = pd.concat([EVT[s_] for s_ in SYMBOLS]).drop_duplicates('family')[['family', 'name']]; FL = {int(f): str(n) for f, n in zip(_nm.family, _nm.name)}; assert sorted(FL) == ALLF, FL
hAE, hX, hT, hF = hits(MODELS[0], PRIM), hits(MODELS[1], PRIM), hits(MODELS[2], PRIM), hits('Fusion', PRIM); hS = hX | hT
hSig = np.concatenate([event_hits(alarm_inj('signal', s, f'r{r}', PRIM), ev) for s, r, ev in SETS])
# ---- genuine burden of the signal (primary budget 2, and descriptive 1 / 4)
BUR = {b: {'all': gen_eps('signal', b), 'BTC': B.eps_per_day(alarm_gen('signal', 'BTCUSDT', b), TV), 'ETH': B.eps_per_day(alarm_gen('signal', 'ETHUSDT', b), TV), 'Sep': gen_eps('signal', b, TVS), 'Oct': gen_eps('signal', b, TVO),
        'AE alone': gen_eps(MODELS[0], b), 'fusion': gen_eps('Fusion', b)} for b in B.BUDGETS}
say('--- genuine TEST-B burden of the UNSEEN PATTERN signal (episodes/day, mean BTC/ETH) ---')
say(pd.DataFrame(BUR).T.round(2).to_string())
# ---- H-U1
HU1 = BUR[PRIM]['all']; h1 = HU1 <= 1.0
say(f'H-U1 signal burden at budget 2 = {HU1:.2f} episodes/day (needs <= 1.0): ' + ('HOLDS' if h1 else 'FAILS'))
# ---- H-U2
d2 = macro_of(hS, OLD) - macro_of(hAE, OLD); bs2 = boot([hS, hAE], OLD, seed=2); dd2 = bs2[:, 0] - bs2[:, 1]; h2 = d2 >= 0.20
say(f'H-U2 known families: recall(XGBoost or Transformer) {pct(macro_of(hS, OLD))} - recall(AE) {pct(macro_of(hAE, OLD))} = {100 * d2:+.1f} points (needs >= +20; bootstrap 95% {100 * np.percentile(dd2, 2.5):+.0f}..{100 * np.percentile(dd2, 97.5):+.0f}): ' + ('HOLDS' if h2 else 'FAILS'))
# ---- H-U3 and the three-way split per family
rows = []
for f in sorted(META.fam.unique()):
    k = (META.fam == f).to_numpy(); n = int(k.sum()); ae_, sp_, fu_, sg_ = hAE[k], hS[k], hF[k], hSig[k]
    rows.append({'family': FL.get(f, f), 'events': n, 'AE only %': 100 * np.mean(ae_ & ~sp_), 'supervised only %': 100 * np.mean(~ae_ & sp_), 'both %': 100 * np.mean(ae_ & sp_), 'neither %': 100 * np.mean(~ae_ & ~sp_),
                 'signal hit %': 100 * np.mean(sg_), 'AE caught': int(ae_.sum()), 'AE caught, fusion missed': int((ae_ & ~fu_).sum()), 'share missed by fusion': (float((ae_ & ~fu_).sum()) / ae_.sum()) if ae_.sum() > 0 else np.nan, 'fusion recall %': 100 * np.mean(fu_)})
FT = pd.DataFrame(rows); say(''); say('--- per family (pooled BTC + ETH, budget 2): who catches the events ---'); say(FT.round(1).to_string(index=False))
use = FT[FT['AE caught'] >= 5]; HU3 = float(use['share missed by fusion'].mean()); h3 = HU3 >= 0.15
say(f'H-U3 macro share of AE-caught events missed by fusion = {pct(HU3)} over {len(use)} families with >= 5 AE-caught events (needs >= 15%): ' + ('HOLDS' if h3 else 'FAILS') + ('; not averaged (fewer than 5 AE-caught events): ' + ', '.join(FT[FT['AE caught'] < 5].family) if (FT['AE caught'] < 5).any() else ''))
# ---- descriptive: recall of the signal, chance level, budgets 1/4, per symbol
rS = macro_of(hSig, ALLF); ch = chance('signal', PRIM, ALLF)
DESC = [{'item': 'signal-hit recall, 8 families, budget 2', 'value': pct(rS)}, {'item': 'chance level of the signal (alarms rolled in time)', 'value': pct(ch)},
        {'item': 'AE recall 8 / old 6 / new 2', 'value': f'{pct(macro_of(hAE, ALLF))} / {pct(macro_of(hAE, OLD))} / {pct(macro_of(hAE, NEW))}'},
        {'item': 'XGBoost-or-Transformer recall 8 / old 6 / new 2', 'value': f'{pct(macro_of(hS, ALLF))} / {pct(macro_of(hS, OLD))} / {pct(macro_of(hS, NEW))}'},
        {'item': 'fusion recall 8 / old 6 / new 2', 'value': f'{pct(macro_of(hF, ALLF))} / {pct(macro_of(hF, OLD))} / {pct(macro_of(hF, NEW))}'},
        {'item': 'signal-hit recall old 6 / new 2', 'value': f'{pct(macro_of(hSig, OLD))} / {pct(macro_of(hSig, NEW))}'}]
for b in [1, 4]:
    hb = np.concatenate([event_hits(alarm_inj('signal', s, f'r{r}', b), ev) for s, r, ev in SETS]); DESC.append({'item': f'signal-hit recall, 8 families, budget {b}', 'value': pct(macro_of(hb, ALLF))})
for sy in SYMBOLS: DESC.append({'item': f'signal-hit recall, {sy}', 'value': pct(macro_of(hSig, ALLF, syms=[sy]))})
say(''); say('--- descriptive (no decision attached) ---'); say(pd.DataFrame(DESC).to_string(index=False))
HYP = {'H-U1': {'text': 'signal burden at budget 2 <= 1.0 genuine episodes/day', 'value': round(HU1, 3), 'holds': bool(h1)}, 'H-U2': {'text': 'known families: recall(XGBoost or Transformer) - recall(AE) >= 0.20', 'value': round(d2, 4), 'ci95': [round(float(np.percentile(dd2, 2.5)), 4), round(float(np.percentile(dd2, 97.5)), 4)], 'holds': bool(h2)},
       'H-U3': {'text': 'macro share of AE-caught events missed by fusion >= 15%', 'value': round(HU3, 4), 'families_used': int(len(use)), 'holds': bool(h3)}}
say(''); say('=== pre-registered hypotheses (protocol U1 v2, Part A) ===')
for k, v in HYP.items(): say(f'{k}: {v["text"]} -> value {v["value"]}: ' + ('HOLDS' if v['holds'] else 'FAILS'))
say(f'{sum(v["holds"] for v in HYP.values())} of 3 hold. Failed hypotheses are reported as failed; nothing was retuned.')

--- genuine TEST-B burden of the UNSEEN PATTERN signal (episodes/day, mean BTC/ETH) ---
    all   BTC   ETH   Sep   Oct  AE alone  fusion
1  1.10  0.95  1.25  1.02  1.18      1.11    0.81
2  2.30  2.25  2.34  2.20  2.39      2.30    1.94
4  2.64  2.92  2.36  2.52  2.76      2.64    3.50
H-U1 signal burden at budget 2 = 2.30 episodes/day (needs <= 1.0): FAILS
H-U2 known families: recall(XGBoost or Transformer) 80.3% - recall(AE) 39.7% = +40.5 points (needs >= +20; bootstrap 95% +37..+45): HOLDS

--- per family (pooled BTC + ETH, budget 2): who catches the events ---
                   family  events  AE only %  supervised only %  both %  neither %  signal hit %  AE caught  AE caught, fusion missed  share missed by fusion  fusion recall %
       pump_and_dump_like      76        2.6               50.0    35.5       11.8          14.5         29                         1                     0.0             86.8
       fake_breakout_like      76        0.0               59.2    22.4       

## 5 · Save

In [ ]:
FT.to_csv(OUT/'u1_partA_family_table.csv', index=False)
RESJ = {'written_utc': datetime.datetime.utcnow().isoformat(timespec='seconds') + 'Z', 'protocol_U1_v2_sha256': PHU, 'library_sha16': T32['library_sha16'], 'test_score_files_sha16': T32['test_score_files_sha16'], 'burden_by_budget': {str(b): {k: round(v, 3) for k, v in d.items()} for b, d in BUR.items()},
        'hypotheses': HYP, 'descriptive': DESC, 'per_family': json.loads(FT.round(4).to_json(orient='records')), 'mock': MOCK, 'note': 'new pre-registered analysis on saved TEST-3 scores (TEST-3 opened once in T0); nothing retuned'}
(OUT/'u1_partA_results.json').write_text(json.dumps(RESJ, indent=1, default=str))
head = '=== notebook 47 · protocol U1 v2 Part A · UNSEEN PATTERN signal (new pre-registered analysis on saved TEST-3 scores) ===\nprotocol U1 v2 sha256 ' + PHU[:16] + '; library ' + T32['library_sha16'] + (' [MOCK]' if MOCK else '')
(OUT/'u1_partA_share_block.txt').write_text(head + '\n' + '\n'.join(LINES[1:]))
say(''); say('Saved: u1_partA_results.json, u1_partA_family_table.csv, u1_partA_share_block.txt in models_b2_v1/ . Send the share block to Claude; it goes into the project docs.')
print(); print((OUT/'u1_partA_share_block.txt').read_text())


Saved: u1_partA_results.json, u1_partA_family_table.csv, u1_partA_share_block.txt in models_b2_v1/ . Send the share block to Claude; it goes into the project docs.

=== notebook 47 · protocol U1 v2 Part A · UNSEEN PATTERN signal (new pre-registered analysis on saved TEST-3 scores) ===
protocol U1 v2 sha256 c448d8d48815aab0; library 3b5d9df23521adf9
saved TEST scores loaded: replicas {'BTCUSDT': [0], 'ETHUSDT': [0]}; TEST-3 events per symbol {'BTCUSDT': 300, 'ETHUSDT': 300}; genuine TEST-B days 61
evaluation helpers ready (same definitions as notebook 32: T1c rolling genuine-series threshold on the injected series, q frozen per symbol and budget, episodes merge bars within 5 minutes)
PASS T01 notebook 32 headline numbers reproduced from the saved scores (macro-8 recall and genuine episodes/day at budget 2 for the three models and fusion)
PASS T02 event hits of the signal equal a plain loop (first TEST-3 set)
PASS T03 signal = AE alarm and not XGBoost alarm and not Transformer alarm (tr